# Exercice 1 : Classification avec Perceptron

L'objectif de cet exercice est d'utiliser un **Perceptron** simple pour apprendre et classifier les sorties des tables logiques de base : **OU (OR)**, **ET (AND)**, et **OU-Exclusif (XOR)**.

In [2]:
from sklearn.linear_model import Perceptron
import numpy as np

#liste d'entrées possibles pour l'opérateur logique
data = [[0, 0], [0, 1], [1, 0], [1, 1]]

## 1. Opérateur OU (OR)

OR labels : `[0, 1, 1, 1]`

In [3]:
#labels correspondants selon liste d'entrées
ORlabels = [0, 1, 1, 1]

#Création du perceptron
operateurOR = Perceptron(max_iter=40,tol=1e-3)
operateurOR.fit(data, ORlabels)

# Prédiction
print("Prédictions OR :", operateurOR.predict(data))

# Évaluation
score_or = operateurOR.score(data, ORlabels)
print(f"Précision (Accuracy) OR : {score_or}")

Prédictions OR : [0 1 1 1]
Précision (Accuracy) OR : 1.0


Le perceptron a bien réussi a prédire les labels à partir des données d'entrées

## 2. Opérateur ET (AND)

**Vérité terrain :** `[0, 0, 0, 1]`

In [4]:
# Labels pour AND
ANDlabels = [0, 0, 0, 1]

# Création et entraînement du Perceptron
operateurAND = Perceptron(max_iter=40,tol=1e-3)
operateurAND.fit(data, ANDlabels)

# Prédiction
print("Prédictions AND :", operateurAND.predict(data))
# Évaluation
score_and = operateurAND.score(data, ANDlabels)
print(f"Précision (Accuracy) AND : {score_and}")

Prédictions AND : [0 0 0 1]
Précision (Accuracy) AND : 1.0


même chose pour l'opérateur ET, le perceptron a réussi a bien prédire les labels à partir des mêmes données d'entrées. CEla vient du fait que ces opérations sont linéaires.

## 3. Opérateur OU-Exclusif (XOR)

**Vérité terrain :** `[0, 1, 1, 0]`

In [5]:
# Labels pour XOR
XORlabels = [0, 1, 1, 0]

# Création et entraînement du Perceptron
operateurXOR = Perceptron(max_iter=40,tol=1e-3)
operateurXOR.fit(data, XORlabels)

# Prédiction
print("Prédictions XOR :", operateurXOR.predict(data))
# Évaluation
score_xor = operateurXOR.score(data, XORlabels)
print(f"Précision (Accuracy) XOR : {score_xor}")

Prédictions XOR : [0 0 0 0]
Précision (Accuracy) XOR : 0.5


Ici la précision est vraiment moins bonne : Parfois 30 ou 50% de réussite. Cela s'explique que XOR n'est pas linéarement séparable, il faut ajouter une couche.

# Exercice d’OU‐Exclusif comme un classifieur
## Questions 4 & 5 : Configuration et Entraînement

Utilisation d'un MLP avec :
- **Activation** : `tanh`
- **Couches cachées** : 1 couche de 2 neurones
- **Max iter** : 10000

In [6]:
from sklearn.neural_network import MLPClassifier

# Définition du modèle
mlp = MLPClassifier(activation='tanh', hidden_layer_sizes=(2,), max_iter=10000, solver='lbfgs', random_state=None)

# Entraînement
mlp.fit(data, XORlabels)

# Prédiction
print("Prédictions MLP XOR :", mlp.predict(data))
print("Score MLP :", mlp.score(data, XORlabels))

# Vérification de la convergence
print(f"Nombre d'itérations : {mlp.n_iter_}")
print(f"Loss : {mlp.loss_}")

Prédictions MLP XOR : [0 0 1 1]
Score MLP : 0.5
Nombre d'itérations : 31
Loss : 0.3484741449311722


### Test de stabilité

In [7]:
scores = []
for i in range(10):
    m = MLPClassifier(activation='tanh', hidden_layer_sizes=(2,), max_iter=5000, solver='lbfgs', random_state=None)
    m.fit(data, XORlabels)
    scores.append(m.score(data, XORlabels))

print(f"Scores sur 10 essais : {scores}")
print(f"Le modèle a réussi {scores.count(1.0)} fois sur 10.")

Scores sur 10 essais : [1.0, 0.5, 0.5, 1.0, 1.0, 0.5, 1.0, 0.5, 0.5, 1.0]
Le modèle a réussi 5 fois sur 10.


### Analyse de la stabilité
Le MLP peut résoudre le XOR (score 1.0), mais l'entraînement est instable. Il y a quelques instances réussies, le reste sont des tentatives où l'algorithme est resté bloqué.

## Question 6 : Extraction des poids et biais

À partir d'un modèle réussi (score = 1.0), Les paramètres du réseau peuvent être extraits.

La topologie est :
- **Entrée** : $x_1, x_2$
- **Cachée** : Neurone A, Neurone B
- **Sortie** : Neurone de sortie

Attributs `sklearn` :
- `coefs_[0]` : Poids Entrée -> Cachée (Matrice 2x2)
- `coefs_[1]` : Poids Cachée -> Sortie (Matrice 2x1)
- `intercepts_[0]` : Biais Cachée (Vecteur 2)
- `intercepts_[1]` : Biais Sortie (Vecteur 1)

In [12]:
final_mlp = None
max_attempts = 100
attempt = 0

while attempt < max_attempts:
    m = MLPClassifier(activation='tanh', hidden_layer_sizes=(2,), max_iter=5000, solver='lbfgs')
    m.fit(data, XORlabels)
    
    if m.score(data, XORlabels) == 1.0:
        final_mlp = m
        break
    attempt += 1

if final_mlp is None:
    print(f"Échec : aucun modèle convergent trouvé après {max_attempts} essais.")
else:
    
    # Extraction des poids
    w1a = final_mlp.coefs_[0][0][0]
    w1b = final_mlp.coefs_[0][0][1]
    w2a = final_mlp.coefs_[0][1][0]
    w2b = final_mlp.coefs_[0][1][1]

    b1 = final_mlp.intercepts_[0][0]
    b2 = final_mlp.intercepts_[0][1]

    w3 = final_mlp.coefs_[1][0][0]
    w4 = final_mlp.coefs_[1][1][0]

    b3 = final_mlp.intercepts_[1][0]

    print("Poids de la couche cachée :")
    print(f"w1a (x1 -> h1) : {w1a:.4f}")
    print(f"w1b (x1 -> h2) : {w1b:.4f}")
    print(f"w2a (x2 -> h1) : {w2a:.4f}")
    print(f"w2b (x2 -> h2) : {w2b:.4f}")
    print(f"b1 (biais h1)  : {b1:.4f}")
    print(f"b2 (biais h2)  : {b2:.4f}")
    print("-" * 20)
    print("Poids de la couche de sortie :")
    print(f"w3 (h1 -> out) : {w3:.4f}")
    print(f"w4 (h2 -> out) : {w4:.4f}")
    print(f"b3 (biais out) : {b3:.4f}")


Poids de la couche cachée :
w1a (x1 -> h1) : -3.1691
w1b (x1 -> h2) : -4.2738
w2a (x2 -> h1) : -2.9896
w2b (x2 -> h2) : -3.8419
b1 (biais h1)  : 1.4396
b2 (biais h2)  : 6.3791
--------------------
Poids de la couche de sortie :
w3 (h1 -> out) : -8.6046
w4 (h2 -> out) : 8.4447
b3 (biais out) : -8.5077


### Analyse des poids

à faire
---

# Exercice 3 : OU-Exclusif comme une Régression


Le réseau doit prédire des valeurs proches de 0 ou 1 

Configuration demandée :
- **Modèle** : `MLPRegressor`
- **Couches cachées** : Au moins 2 couches
- **Activation** : `tanh`
- **Solver** : `lbfgs`

In [14]:
from sklearn.neural_network import MLPRegressor

# Définition du régresseur
regressor = MLPRegressor(hidden_layer_sizes=(4, 4), activation='tanh', solver='lbfgs', max_iter=5000, random_state=42)

# Entraînement
regressor.fit(data, XORlabels)

# Prédiction
predictions_reg = regressor.predict(data)
print("Prédictions Régression XOR :", predictions_reg)
print("Valeurs attendues :", XORlabels)

# Évaluation
# 1.0 est le meilleur score possible
score_reg = regressor.score(data, XORlabels)
print(f"Score R^2 : {score_reg}")
print(f"Convergence en {regressor.n_iter_} itérations")

Prédictions Régression XOR : [2.10822409e-04 9.99898849e-01 9.99802499e-01 3.11142609e-04]
Valeurs attendues : [0, 1, 1, 0]
Score R^2 : 0.9999998095058562
Convergence en 125 itérations
